In [6]:
import requests
import pandas as pd
import kaleido
from pathlib import Path
BASE_DIR = Path.cwd()  # project root


url_spend = "https://api.worldbank.org/v2/country/all/indicator/SH.XPD.GHED.PP.CD?date=2010:2019&format=json&per_page=20000"
url_life  = "https://api.worldbank.org/v2/country/all/indicator/SP.DYN.LE00.IN?date=2010:2019&format=json&per_page=20000"

def fetch_wb(url):
    r = requests.get(url, timeout=30) 
    r.raise_for_status()              
    return r.json()[1]



In [7]:
print(BASE_DIR)

/Users/nataliam/Desktop/All/python_projects/aboba


In [8]:
spend_raw = fetch_wb(url_spend)
life_raw  = fetch_wb(url_life)


def to_df(raw, value_col):
    rows = [{"country": d["country"]["value"],
             "iso":     d["countryiso3code"],
             "year":    int(d["date"]),
             value_col: d["value"]} for d in raw]
    return pd.DataFrame(rows)

df_spend = to_df(spend_raw, "health_spend")
df_life  = to_df(life_raw,  "life_exp")




In [9]:
eu_iso = ["AUT","BEL","BGR","HRV","CYP","CZE","DNK","EST","FIN",
          "FRA","DEU","GRC","HUN","IRL","ITA","LVA","LTU","LUX",
          "MLT","NLD","POL","PRT","ROU","SVK","SVN","ESP","SWE"]


eu_spend = df_spend[df_spend["iso"].isin(eu_iso)]

eu_life = df_life[df_life["iso"].isin(eu_iso)]

In [10]:
eu_spend.head()

,country,iso,year,health_spend
600,Austria,AUT,2019,4771.374180
601,Austria,AUT,2018,4401.698591
602,Austria,AUT,2017,4168.014161
603,Austria,AUT,2016,3971.299362
604,Austria,AUT,2015,3778.943647


In [11]:
eu_life.head()

,country,iso,year,life_exp
600,Austria,AUT,2019,81.895122
601,Austria,AUT,2018,81.692683
602,Austria,AUT,2017,81.643902
603,Austria,AUT,2016,81.641463
604,Austria,AUT,2015,81.190244


In [12]:
key_years = [2010, 2015, 2019]


In [13]:
eu_spend_keys = eu_spend[eu_spend["year"].isin(key_years)]
eu_life_keys  = eu_life[eu_life["year"].isin(key_years)]

In [14]:
dupes = eu_spend_keys[eu_spend_keys.duplicated(subset=["iso", "year"], keep=False)]
print(dupes)

Empty DataFrame
Columns: [country, iso, year, health_spend]
Index: []


In [15]:
spend_pivot = eu_spend_keys.pivot_table(index="iso", columns="year", values="health_spend", aggfunc="mean")
life_pivot  = eu_life_keys.pivot_table(index="iso", columns="year", values="life_exp", aggfunc="mean")

In [16]:
spend_pivot = spend_pivot.dropna()
life_pivot  = life_pivot.dropna()

In [17]:
spend_pivot.head()

year,2010,2015,2019
iso,,,
AUT,3114.108095,3778.943647,4771.374180
BEL,3113.644669,3822.866962,4598.111747
BGR,584.071062,776.106471,1121.080448
CYP,1008.904367,898.312756,1694.379266
CZE,1789.428864,2099.511776,2893.933067


In [18]:
life_pivot.head()

year,2010,2015,2019
iso,,,
AUT,80.580488,81.190244,81.895122
BEL,80.182927,80.992683,81.995122
BGR,73.512195,74.614634,75.112195
CYP,80.632000,81.464000,81.454000
CZE,77.424390,78.578049,79.229268


In [19]:


spend_pivot.columns = ["spend_2010", "spend_2015", "spend_2019"]
life_pivot.columns  = ["life_2010",  "life_2015",  "life_2019"]

df = spend_pivot.join(life_pivot, how="inner")  
print(df.shape)   


(27, 6)


In [20]:
print(df.head())

      spend_2010   spend_2015   spend_2019  life_2010  life_2015  life_2019
iso                                                                        
AUT  3114.108095  3778.943647  4771.374180  80.580488  81.190244  81.895122
BEL  3113.644669  3822.866962  4598.111747  80.182927  80.992683  81.995122
BGR   584.071062   776.106471  1121.080448  73.512195  74.614634  75.112195
CYP  1008.904367   898.312756  1694.379266  80.632000  81.464000  81.454000
CZE  1789.428864  2099.511776  2893.933067  77.424390  78.578049  79.229268


In [21]:

# % change in spending from 2010 to 2015
df["spend_change_pct"] = ((df["spend_2015"] - df["spend_2010"]) / df["spend_2010"]) * 100

# did spending go up or down?
df["spend_group"] = df["spend_change_pct"].apply(
    lambda x: "Increased" if x > 0 else "Decreased"
)


In [22]:

iso_to_name = {d["countryiso3code"]: d["country"]["value"] for d in spend_raw}
df["country"] = df.index.map(iso_to_name)

print(df[["country", "spend_change_pct", "spend_group", "life_2015", "life_2019"]])

             country  spend_change_pct spend_group  life_2015  life_2019
iso                                                                     
AUT          Austria         21.349148   Increased  81.190244  81.895122
BEL          Belgium         22.777882   Increased  80.992683  81.995122
BGR         Bulgaria         32.878775   Increased  74.614634  75.112195
CYP           Cyprus        -10.961555   Decreased  81.464000  81.454000
CZE          Czechia         17.328597   Increased  78.578049  79.229268
DEU          Germany         26.225562   Increased  80.641463  81.292683
DNK          Denmark         11.250229   Increased  80.702439  81.451220
ESP            Spain          5.533407   Increased  82.831707  83.831707
EST          Estonia         38.126225   Increased  77.590244  78.646341
FIN          Finland         15.171376   Increased  81.480488  81.982927
FRA           France         19.488903   Increased  82.321951  82.826829
GRC           Greece        -31.493395   Decreased 

In [23]:
df[["life_2015", "life_2019"]] = df[["life_2015", "life_2019"]].round(2)

In [24]:
df.loc[df["country"] == "Croatia", "country"] = "Croatia*"

In [25]:
import plotly.graph_objects as go

colors = {"Increased": "#2ecc71", "Decreased": "#e74c3c"}



In [26]:
fig = go.Figure()  

legend_shown = {"Increased": False, "Decreased": False}

for _, row in df.sort_values("life_2015").iterrows():
    color = colors[row["spend_group"]]
    group = row["spend_group"]
    show = not legend_shown[group]

    fig.add_trace(go.Scatter(
        x=[row["life_2015"], row["life_2019"]],
        y=[row["country"], row["country"]],
        mode="lines",
        line=dict(color=color, width=2),
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[row["life_2015"]],
        y=[row["country"]],
        mode="markers",
        marker=dict(color="white", size=10, line=dict(color=color, width=2)),
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=[row["life_2019"]],
        y=[row["country"]],
        mode="markers",
        marker=dict(color=color, size=10),
        name=f"Spending {group} (2010→2015)",
        showlegend=show,
        hovertemplate=f"<b>{row['country']}</b><br>2015: {row['life_2015']:.1f}<br>2019: {row['life_2019']:.1f}<br>Spending: {row['spend_change_pct']:+.1f}%<extra></extra>"
    ))

    legend_shown[group] = True

fig.update_layout(
    #title=dict(
        #text="More healthcare expenses - longer lives? On average - yes.<br><sup> The countries that increased healthcare spending gained on average 0.84 years <br> of life expectancy between 2015 and 2019 — nearly twice the gain of countries that cut spending.</sup>",
        #x=0.5,
        #xanchor="center"
    #),
    title=dict(
    text="More healthcare expenses - longer lives? On average - yes.",
    x=0.5,
    xanchor="center",
    font=dict(color="#1a2f4a")  # your dark blue
),
    xaxis_title="Life Expectancy (years)",
    yaxis_title="",
    height=800,
    plot_bgcolor="white",
    xaxis=dict(
        range=[df[["life_2015", "life_2019"]].min().min() - 0.5,
               df[["life_2015", "life_2019"]].max().max() + 0.5],
        gridcolor="#eeeeee"
    ),
    legend=dict(
        x=1.48,
        y=0.98,
        xanchor="right",
        yanchor="top"
    ),
)

# Annotation 1: source
fig.add_annotation(
    text="Source: World Bank,<br>Domestic general government<br>health expenditure (% of GDP),<br>Life expectancy at birth",
    xref="paper", yref="paper",
    x=1.4, y=-0.1,
    showarrow=False,
    font=dict(size=10, color="darkgrey"),
    xanchor="right",
    bordercolor="grey",
    borderwidth=1,
    borderpad=6,
    bgcolor="white"
)


# Annotation 2: Croatia note
fig.add_annotation(
    text="*Croatia joined the EU in 2013.        <br>Gains may reflect accession reforms        <br>rather than spending trends.        ",
    xref="paper", yref="paper",
    x=1.47, y=0.8,
    showarrow=False,
    font=dict(size=11, color="#4a6080"),
    xanchor="right",
    bgcolor="rgba(245, 241, 248, 0.85)",
    bordercolor="rgba(0,0,0,0)",
    borderpad=8
)

# Annotation 3: subtitle
fig.add_annotation(
    text="The EU countries that increased healthcare spending gained on average 0.84 years<br>of life expectancy between 2015 and 2019 — nearly twice the gain of countries that cut spending.",
    xref="paper", yref="paper",
    x=0.6, y=1.06,
    showarrow=False,
    font=dict(size=13, color="#4a6080"),
    xanchor="center",
    yanchor="top",
    bgcolor="rgba(240, 244, 248, 0.85)",
    bordercolor="rgba(0,0,0,0)",
    borderpad=8
)


                   
    
fig.show()
    
#More healthcare expenses - longer lives? On average - yes.
#The few EU countries that cut healthcare spending between 2010 and 2015 saw smaller life expectancy gains five year later. 


In [27]:

df["life_change"] = df["life_2019"] - df["life_2015"]

print(df.groupby("spend_group")[["life_change"]].agg(["mean", "min", "max", "count"]).round(2))


            life_change                  
                   mean   min   max count
spend_group                              
Decreased          0.49 -0.01  0.94     5
Increased          0.84  0.45  1.96    22


In [28]:
print(df[["country", "spend_group", "life_change"]].sort_values(["spend_group", "life_change"], ascending=[True, False]))

             country spend_group  life_change
iso                                          
HRV         Croatia*   Decreased         0.94
GRC           Greece   Decreased         0.60
PRT         Portugal   Decreased         0.56
LUX       Luxembourg   Decreased         0.35
CYP           Cyprus   Decreased        -0.01
LTU        Lithuania   Increased         1.96
IRL          Ireland   Increased         1.25
SVK  Slovak Republic   Increased         1.11
EST          Estonia   Increased         1.06
BEL          Belgium   Increased         1.01
ESP            Spain   Increased         1.00
ITA            Italy   Increased         0.96
LVA           Latvia   Increased         0.91
SWE           Sweden   Increased         0.91
MLT            Malta   Increased         0.86
DNK          Denmark   Increased         0.75
HUN          Hungary   Increased         0.75
SVN         Slovenia   Increased         0.75
AUT          Austria   Increased         0.71
ROU          Romania   Increased  

In [29]:
fig.write_html("in_five_years_chart.html")

In [30]:
fig.to_html(full_html=False, include_plotlyjs='cdn')

'<div>                        <script>window.PlotlyConfig = {MathJaxConfig: \'local\'};</script>\n        <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.4.0.min.js" integrity="sha256-KEmPoupLpFyGMyGAiOsiNDbKDKAvxXAn/W+oQa0ZAfk=" crossorigin="anonymous"></script>                <div id="bb383789-46fe-41a3-8eb1-f2d0a2b62adf" class="plotly-graph-div" style="height:800px; width:100%;"></div>            <script>                window.PLOTLYENV=window.PLOTLYENV || {};                                if (document.getElementById("bb383789-46fe-41a3-8eb1-f2d0a2b62adf")) {                    Plotly.newPlot(                        "bb383789-46fe-41a3-8eb1-f2d0a2b62adf",                        [{"line":{"color":"#2ecc71","width":2},"mode":"lines","showlegend":false,"x":[74.32,76.28],"y":["Lithuania","Lithuania"],"type":"scatter"},{"marker":{"color":"white","line":{"color":"#2ecc71","width":2},"size":10},"mode":"markers","showlegend":false,"x":[74.32],"y":["Lithuania"],"type":"scatter"},{

In [31]:
# Static PNG — for embedding in README
fig.write_image(
    BASE_DIR/"outputs"/"in_five_years.png",
    width=1200, height=600, scale=2,
)

# Interactive HTML — for the grader to hover/zoom
fig.write_html(
    BASE_DIR/"outputs"/"in_five_years_interact.html"
)

/var/folders/jq/wxn09tld1nlgws0gvk_bcwz00000gn/T/ipykernel_79824/3799917453.py:2: DeprecationWarning: 
Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).

  fig.write_image(
